# NOTEBOOK 2: LIMPIEZA, ESTANDARIZACIÓN Y ETL

---

## Proyecto: Arquitectura de BI y Big Data para Análisis del Turismo Académico en Medellín

**Objetivo del Notebook:** Limpiar, estandarizar y consolidar los datos de las tres universidades en un único dataset de alta calidad listo para análisis.

---

### Contenido:
1. Carga de datos crudos
2. Funciones de limpieza reutilizables
3. Eliminación de valores nulos y duplicados
4. Estandarización de nomenclaturas
5. Creación de variables derivadas
6. Validación de calidad post-limpieza
7. Consolidación y exportación

---
## 1. CONFIGURACIÓN E IMPORTACIÓN

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
from datetime import datetime

# Configuración
# Como el notebook está en 'notebooks/', subimos un nivel para llegar a la raíz
BASE_DIR = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
DATA_RAW_DIR = BASE_DIR / 'data' / 'raw'
DATA_PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
REPORTS_DIR = BASE_DIR / 'outputs' / 'reportes'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Configuración completada")
print(f"Directorio base: {BASE_DIR}")
print(f"Datos crudos: {DATA_RAW_DIR}")
print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Configuración completada
Directorio base: C:\Users\Andres Diaz\Desktop\pipeline-proyecto-grado
Datos crudos: C:\Users\Andres Diaz\Desktop\pipeline-proyecto-grado\data\raw
Fecha de ejecución: 2025-10-31 01:30:49


---
## 2. FUNCIONES DE LIMPIEZA REUTILIZABLES

In [2]:
def limpiar_espacios(df, columnas):
    """
    Elimina espacios en blanco adicionales en columnas de texto.
    """
    df_clean = df.copy()
    for col in columnas:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str).str.strip()
    return df_clean


def eliminar_filas_vacias(df):
    """
    Elimina filas completamente vacías.
    """
    antes = len(df)
    df_clean = df.dropna(how='all')
    despues = len(df_clean)
    eliminadas = antes - despues
    
    print(f"  - Filas eliminadas (vacías): {eliminadas:,}")
    print(f"  - Filas restantes: {despues:,}")
    
    return df_clean


def estandarizar_paises(df, columna='PAIS_EXTRANJERO'):
    """
    Estandariza nombres de países a nomenclatura oficial.
    """
    diccionario_paises = {
        'ESTADOS UNIDOS DE AMÉRICA': 'Estados Unidos',
        'ESTADOS UNIDOS': 'Estados Unidos',
        'USA': 'Estados Unidos',
        'US': 'Estados Unidos',
        'MÉXICO': 'México',
        'MEXICO': 'México',
        'ESPAÑA': 'España',
        'BRASIL': 'Brasil',
        'ITALY': 'Italia',
        'ITALIA': 'Italia',
        'GRECIA': 'Grecia',
        'GREECE': 'Grecia',
        'TURQUÍA': 'Turquía',
        'TURKEY': 'Turquía',
        'RUMANÍA': 'Rumania',
        'ROMANIA': 'Rumania',
        'VENEZUELA': 'Venezuela'
    }
    
    df_clean = df.copy()
    if columna in df_clean.columns:
        df_clean[columna] = df_clean[columna].str.upper().str.strip()
        df_clean[columna] = df_clean[columna].replace(diccionario_paises)
        
        paises_unicos_antes = df[columna].nunique()
        paises_unicos_despues = df_clean[columna].nunique()
        print(f"  - Países únicos antes: {paises_unicos_antes}")
        print(f"  - Países únicos después: {paises_unicos_despues}")
    
    return df_clean


def convertir_tipos_datos(df):
    """
    Convierte columnas a los tipos de datos apropiados.
    """
    df_clean = df.copy()
    
    # Convertir columnas numéricas
    columnas_numericas = ['AÑO', 'SEMESTRE', 'NUM_DIAS_MOVILIDAD', 
                          'VALOR_FINANCIACION_NACIONAL', 'VALOR_FINANCIACION_INTERNAC',
                          'ID_PAIS_EXTRANJERO', 'ID_TIPO_MOV_EST_EXTRANJ']
    
    for col in columnas_numericas:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    return df_clean


def crear_variables_derivadas(df):
    """
    Crea nuevas variables útiles para el análisis.
    """
    df_enhanced = df.copy()
    
    # 1. Periodo completo (Año-Semestre)
    if 'AÑO' in df_enhanced.columns and 'SEMESTRE' in df_enhanced.columns:
        df_enhanced['PERIODO'] = df_enhanced['AÑO'].astype(str) + '-' + df_enhanced['SEMESTRE'].astype(str)
    
    # 2. Nombre completo del estudiante
    if all(col in df_enhanced.columns for col in ['PRIMER_NOMBRE', 'PRIMER_APELLIDO']):
        df_enhanced['NOMBRE_COMPLETO'] = (
            df_enhanced['PRIMER_NOMBRE'].fillna('') + ' ' + 
            df_enhanced['SEGUNDO_NOMBRE'].fillna('') + ' ' + 
            df_enhanced['PRIMER_APELLIDO'].fillna('') + ' ' + 
            df_enhanced['SEGUNDO_APELLIDO'].fillna('')
        ).str.strip().str.replace(r'\s+', ' ', regex=True)
    
    # 3. Categoría de duración de movilidad
    if 'NUM_DIAS_MOVILIDAD' in df_enhanced.columns:
        def categorizar_duracion(dias):
            if pd.isna(dias):
                return 'No especificado'
            elif dias <= 7:
                return 'Corta (≤7 días)'
            elif dias <= 30:
                return 'Media (8-30 días)'
            elif dias <= 90:
                return 'Larga (31-90 días)'
            else:
                return 'Muy larga (>90 días)'
        
        df_enhanced['CATEGORIA_DURACION'] = df_enhanced['NUM_DIAS_MOVILIDAD'].apply(categorizar_duracion)
    
    # 4. Financiación total
    if 'VALOR_FINANCIACION_NACIONAL' in df_enhanced.columns and 'VALOR_FINANCIACION_INTERNAC' in df_enhanced.columns:
        df_enhanced['FINANCIACION_TOTAL'] = (
            df_enhanced['VALOR_FINANCIACION_NACIONAL'].fillna(0) + 
            df_enhanced['VALOR_FINANCIACION_INTERNAC'].fillna(0)
        )
    
    # 5. Tiene convenio (booleano)
    if 'MOVILIDAD_POR_CONVENIO' in df_enhanced.columns:
        df_enhanced['TIENE_CONVENIO'] = df_enhanced['MOVILIDAD_POR_CONVENIO'].isin(['S', 'SI', 'Sí', 'Y', 'Yes'])
    
    print(f"  - Variables derivadas creadas: {['PERIODO', 'NOMBRE_COMPLETO', 'CATEGORIA_DURACION', 'FINANCIACION_TOTAL', 'TIENE_CONVENIO']}")
    
    return df_enhanced


print("✓ Funciones de limpieza definidas")

✓ Funciones de limpieza definidas


In [ ]:
# === Capa de lectura flexible y normalización de esquema ===
import json
import unicodedata

CONFIG_PATH = BASE_DIR / 'config' / 'column_mappings.json'

# Utilidad: normalizar nombre de columna (quitar tildes, espacios, mayúsculas)
def _norm_col(name: str) -> str:
    if not isinstance(name, str):
        name = str(name)
    # Quitar acentos
    name = ''.join(c for c in unicodedata.normalize('NFD', name) if unicodedata.category(c) != 'Mn')
    name = name.strip().upper().replace(' ', '_')
    return name

# Leer archivo (CSV o Excel) en DataFrame
def leer_archivo_flexible(path: Path, **kwargs):
    suf = path.suffix.lower()
    if suf in ['.xlsx', '.xls']:
        return pd.read_excel(path, sheet_name=kwargs.pop('sheet_name', 0), dtype=kwargs.pop('dtype', None))
    elif suf == '.csv':
        return pd.read_csv(path, **kwargs)
    else:
        raise ValueError(f"Formato no soportado: {suf}")

# Cargar configuración de mapeo
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    _cfg = json.load(f)
CANONICAL_COLS = _cfg['canonical_columns']
SOURCES_MAP = _cfg['sources']

# Encontrar mejor match de columna origen -> canónica
def _match_column(df_cols_norm, candidates):
    # candidates: lista de posibles nombres (raw), los normalizamos y buscamos
    cand_norm = [_norm_col(c) for c in candidates]
    for c in cand_norm:
        if c in df_cols_norm:
            return c
    return None

# Aplicar mapeo a columnas hacia esquema canónico
def mapear_columnas(df: pd.DataFrame, source_key: str) -> pd.DataFrame:
    df2 = df.copy()
    # Normalizar encabezados
    df2.columns = [_norm_col(c) for c in df2.columns]

    mapping = SOURCES_MAP.get(source_key, {})
    renames = {}
    for canon in CANONICAL_COLS:
        candidates = mapping.get(canon, [canon])
        match = _match_column(set(df2.columns), candidates)
        if match is not None:
            renames[match] = canon
    if renames:
        df2 = df2.rename(columns=renames)

    # Asegurar todas las columnas canónicas existan
    for col in CANONICAL_COLS:
        if col not in df2.columns:
            df2[col] = pd.NA

    # Tipificar campos comunes
    # AÑO y SEMESTRE a int si posible
    for col in ['AÑO', 'SEMESTRE']:
        if col in df2.columns:
            df2[col] = pd.to_numeric(df2[col], errors='coerce').astype('Int64')

    # NUM_DIAS_MOVILIDAD y FINANCIACION_TOTAL a numérico
    for col in ['NUM_DIAS_MOVILIDAD', 'FINANCIACION_TOTAL']:
        if col in df2.columns:
            df2[col] = pd.to_numeric(df2[col], errors='coerce')

    # Si no hay FINANCIACION_TOTAL, intentar calcularla como suma de columnas parciales
    if ('FINANCIACION_TOTAL' not in df2.columns) or (df2['FINANCIACION_TOTAL'].isna().all()):
        parciales = []
        for c in ['VALOR_FINANCIACION_NACIONAL', 'VALOR_FINANCIACION_INTERNAC', 'VALOR_FINANCIACION_INTERNACIONAL', 'FINANCIACION_NACIONAL', 'FINANCIACION_INTERNACIONAL']:
            cn = _norm_col(c)
            if cn in df2.columns:
                df2[cn] = pd.to_numeric(df2[cn], errors='coerce')
                parciales.append(cn)
        if parciales:
            df2['FINANCIACION_TOTAL'] = sum(df2[c] for c in parciales)

    # Derivar CATEGORIA_DURACION si falta
    if 'CATEGORIA_DURACION' in df2.columns:
        if df2['CATEGORIA_DURACION'].isna().all() and 'NUM_DIAS_MOVILIDAD' in df2.columns:
            def _cat_dias(x):
                try:
                    x = float(x)
                except Exception:
                    return pd.NA
                if pd.isna(x):
                    return pd.NA
                if x <= 30:
                    return 'CORTA'
                elif x <= 90:
                    return 'MEDIA'
                else:
                    return 'LARGA'
            df2['CATEGORIA_DURACION'] = df2['NUM_DIAS_MOVILIDAD'].apply(_cat_dias)

    # UNIVERSIDAD, PAIS_EXTRANJERO, INSTITUCION_EXTRANJERA, TIPO_MOV_EST_EXTRANJ a string
    for col in ['UNIVERSIDAD', 'PAIS_EXTRANJERO', 'INSTITUCION_EXTRANJERA', 'TIPO_MOV_EST_EXTRANJ', 'CATEGORIA_DURACION']:
        if col in df2.columns:
            df2[col] = df2[col].astype('string')

    return df2[CANONICAL_COLS]

# Loader de una fuente con clave y ruta
def cargar_fuente(path: Path, source_key: str, nombre_mostrar: str):
    print(f"\n{'='*60}\nCargando fuente: {nombre_mostrar}\n{'='*60}")
    try:
        df_raw = leer_archivo_flexible(path, na_values=['#N/A', 'N/A', 'NA', '', ' '], keep_default_na=True)
        print(f"✓ Archivo leído: {path.name} | Filas: {len(df_raw):,} | Columnas: {len(df_raw.columns)}")
        print(f"Encabezados originales: {list(df_raw.columns)[:10]}{'...' if len(df_raw.columns)>10 else ''}")
        df_norm = mapear_columnas(df_raw, source_key)
        df_norm['UNIVERSIDAD'] = df_norm['UNIVERSIDAD'].fillna(nombre_mostrar)
        print(f"✓ Esquema normalizado a columnas canónicas: {len(df_norm.columns)} columnas")
        return df_norm
    except FileNotFoundError:
        print(f"✗ No se encontró el archivo: {path}")
        return None
    except Exception as e:
        print(f"✗ Error cargando fuente {nombre_mostrar}: {e}")
        return None

---
## 3. CARGA Y LIMPIEZA DE DATOS - IUSH

In [3]:
print("="*80)
print("PROCESAMIENTO ETL - IUSH")
print("="*80)

# Cargar datos
df_iush_raw = pd.read_csv(
    DATA_RAW_DIR / 'iush.csv',
    na_values=['#N/A', 'N/A', 'NA', '', ' '],
    keep_default_na=True
)

print(f"\n1. DATOS CRUDOS CARGADOS")
print(f"  - Registros: {len(df_iush_raw):,}")

# Paso 1: Eliminar filas vacías
print(f"\n2. ELIMINANDO FILAS VACÍAS")
df_iush_clean = eliminar_filas_vacias(df_iush_raw)

# Paso 2: Limpiar espacios en columnas de texto
print(f"\n3. LIMPIANDO ESPACIOS EN TEXTO")
columnas_texto = ['PRIMER_NOMBRE', 'SEGUNDO_NOMBRE', 'PRIMER_APELLIDO', 'SEGUNDO_APELLIDO',
                  'PAIS_EXTRANJERO', 'INSTITUCION_EXTRANJERA', 'TIPO_MOV_EST_EXTRANJ']
df_iush_clean = limpiar_espacios(df_iush_clean, columnas_texto)
print(f"  - Espacios eliminados en {len(columnas_texto)} columnas")

# Paso 3: Estandarizar nombres de países
print(f"\n4. ESTANDARIZANDO NOMBRES DE PAÍSES")
df_iush_clean = estandarizar_paises(df_iush_clean)

# Paso 4: Convertir tipos de datos
print(f"\n5. CONVIRTIENDO TIPOS DE DATOS")
df_iush_clean = convertir_tipos_datos(df_iush_clean)
print(f"  - Tipos de datos convertidos correctamente")

# Paso 5: Eliminar duplicados
print(f"\n6. ELIMINANDO DUPLICADOS")
antes_duplicados = len(df_iush_clean)
df_iush_clean = df_iush_clean.drop_duplicates()
duplicados_eliminados = antes_duplicados - len(df_iush_clean)
print(f"  - Duplicados eliminados: {duplicados_eliminados}")

# Paso 6: Crear variables derivadas
print(f"\n7. CREANDO VARIABLES DERIVADAS")
df_iush_clean = crear_variables_derivadas(df_iush_clean)

# Agregar identificador de universidad
df_iush_clean['UNIVERSIDAD'] = 'IUSH'

print(f"\n" + "="*80)
print(f"RESUMEN LIMPIEZA - IUSH")
print(f"="*80)
print(f"Registros iniciales: {len(df_iush_raw):,}")
print(f"Registros finales: {len(df_iush_clean):,}")
print(f"Reducción: {len(df_iush_raw) - len(df_iush_clean):,} ({((len(df_iush_raw) - len(df_iush_clean))/len(df_iush_raw)*100):.2f}%)")

PROCESAMIENTO ETL - IUSH

1. DATOS CRUDOS CARGADOS
  - Registros: 767

2. ELIMINANDO FILAS VACÍAS
  - Filas eliminadas (vacías): 740
  - Filas restantes: 27

3. LIMPIANDO ESPACIOS EN TEXTO
  - Espacios eliminados en 7 columnas

4. ESTANDARIZANDO NOMBRES DE PAÍSES
  - Países únicos antes: 9
  - Países únicos después: 9

5. CONVIRTIENDO TIPOS DE DATOS
  - Tipos de datos convertidos correctamente

6. ELIMINANDO DUPLICADOS
  - Duplicados eliminados: 0

7. CREANDO VARIABLES DERIVADAS
  - Variables derivadas creadas: ['PERIODO', 'NOMBRE_COMPLETO', 'CATEGORIA_DURACION', 'FINANCIACION_TOTAL', 'TIENE_CONVENIO']

RESUMEN LIMPIEZA - IUSH
Registros iniciales: 767
Registros finales: 27
Reducción: 740 (96.48%)


In [ ]:
# === Carga de fuentes con esquema variable ===
# 1) IUSH (CSV)
ruta_iush = DATA_RAW_DIR / 'iush.csv'
df_iush_norm = cargar_fuente(ruta_iush, 'iush', 'IUSH')

# 2) Docentes Exterior (Excel) - opcional
ruta_docentes = DATA_RAW_DIR / '2. Movilidad_de_docentes_del_exterior_hacia_colombia.xlsx'
df_docentes_norm = cargar_fuente(ruta_docentes, 'docentes_exterior', 'DOCENTES_EXTERIOR')

# Unificación (si ambas existen) - opcional
frames = [df for df in [df_iush_norm, df_docentes_norm] if df is not None]
if frames:
    df_consolidado = pd.concat(frames, ignore_index=True)
    print(f"\n✓ Consolidado creado: {len(df_consolidado):,} registros")
    # Guardar para uso posterior
    out_path = DATA_PROCESSED_DIR / 'datos_consolidados_limpios.csv'
    df_consolidado.to_csv(out_path, index=False)
    print(f"✓ Guardado: {out_path}")
else:
    print("\n⚠ No se pudieron cargar fuentes para consolidar")

---
## 4. VALIDACIÓN DE CALIDAD POST-LIMPIEZA

In [4]:
print("\n" + "="*80)
print("VALIDACIÓN DE CALIDAD DE DATOS")
print("="*80)

# 1. Estadísticas de completitud
print("\n1. COMPLETITUD DE DATOS")
print("-" * 80)

completitud = pd.DataFrame({
    'Total_Registros': len(df_iush_clean),
    'Valores_Nulos': df_iush_clean.isnull().sum(),
    'Valores_Completos': len(df_iush_clean) - df_iush_clean.isnull().sum(),
    'Porcentaje_Completitud': ((len(df_iush_clean) - df_iush_clean.isnull().sum()) / len(df_iush_clean) * 100).round(2)
}).sort_values('Porcentaje_Completitud')

print(completitud.head(10))

# 2. Verificación de rangos válidos
print("\n2. VALIDACIÓN DE RANGOS")
print("-" * 80)

validaciones = []

# Años válidos (2020-2025)
if 'AÑO' in df_iush_clean.columns:
    años_validos = df_iush_clean['AÑO'].between(2020, 2025).sum()
    validaciones.append(f"✓ Años válidos (2020-2025): {años_validos}/{len(df_iush_clean)}")

# Semestres válidos (1-2)
if 'SEMESTRE' in df_iush_clean.columns:
    semestres_validos = df_iush_clean['SEMESTRE'].isin([1, 2]).sum()
    validaciones.append(f"✓ Semestres válidos (1-2): {semestres_validos}/{len(df_iush_clean)}")

# Días de movilidad positivos
if 'NUM_DIAS_MOVILIDAD' in df_iush_clean.columns:
    dias_positivos = (df_iush_clean['NUM_DIAS_MOVILIDAD'] > 0).sum()
    validaciones.append(f"✓ Días de movilidad positivos: {dias_positivos}/{len(df_iush_clean)}")

for val in validaciones:
    print(val)

# 3. Distribución de categorías
print("\n3. DISTRIBUCIÓN POR CATEGORÍAS")
print("-" * 80)

if 'PAIS_EXTRANJERO' in df_iush_clean.columns:
    print(f"\nPaíses únicos: {df_iush_clean['PAIS_EXTRANJERO'].nunique()}")
    print(f"Top 5 países:")
    print(df_iush_clean['PAIS_EXTRANJERO'].value_counts().head())

if 'TIPO_MOV_EST_EXTRANJ' in df_iush_clean.columns:
    print(f"\nTipos de movilidad:")
    print(df_iush_clean['TIPO_MOV_EST_EXTRANJ'].value_counts())

if 'CATEGORIA_DURACION' in df_iush_clean.columns:
    print(f"\nCategorías de duración:")
    print(df_iush_clean['CATEGORIA_DURACION'].value_counts())


VALIDACIÓN DE CALIDAD DE DATOS

1. COMPLETITUD DE DATOS
--------------------------------------------------------------------------------
                             Total_Registros  Valores_Nulos  \
VALOR_FINANCIACION_NACIONAL               27             27   
ID_FUENTE_NACIONAL_INVESTIG               27             27   
FUENTE_NACIONAL_INVESTIG                  27             27   
CODIGO_CONVENIO                           27             27   
AÑO                                       27              0   
FINANCIACION_TOTAL                        27              0   
CATEGORIA_DURACION                        27              0   
NOMBRE_COMPLETO                           27              0   
PERIODO                                   27              0   
VALOR_FINANCIACION_INTERNAC               27              0   

                             Valores_Completos  Porcentaje_Completitud  
VALOR_FINANCIACION_NACIONAL                  0                     0.0  
ID_FUENTE_NACIONAL_INV

---
## 5. CONSOLIDACIÓN DE DATOS (PREPARADO PARA 3 UNIVERSIDADES)

In [5]:
# Por ahora solo tenemos IUSH, pero el código está preparado para consolidar
print("\n" + "="*80)
print("CONSOLIDACIÓN DE DATOS")
print("="*80)

# Lista de DataFrames a consolidar
dataframes = [df_iush_clean]
nombres_unis = ['IUSH']

# Cuando tengamos datos de UdeA y UNAC, agregaremos:
# if df_udea_clean is not None:
#     dataframes.append(df_udea_clean)
#     nombres_unis.append('Universidad de Antioquia')
# if df_unac_clean is not None:
#     dataframes.append(df_unac_clean)
#     nombres_unis.append('UNAC')

# Consolidar todos los datos
df_consolidado = pd.concat(dataframes, ignore_index=True)

print(f"\nUniversidades consolidadas: {', '.join(nombres_unis)}")
print(f"Total de registros: {len(df_consolidado):,}")
print(f"\nDistribución por universidad:")
print(df_consolidado['UNIVERSIDAD'].value_counts())


CONSOLIDACIÓN DE DATOS

Universidades consolidadas: IUSH
Total de registros: 27

Distribución por universidad:
UNIVERSIDAD
IUSH    27
Name: count, dtype: int64


---
## 6. EXPORTACIÓN DE DATOS LIMPIOS

In [6]:
# Exportar a CSV
archivo_salida = DATA_PROCESSED_DIR / 'datos_consolidados_limpios.csv'
df_consolidado.to_csv(archivo_salida, index=False, encoding='utf-8')

print(f"\n✓ Datos consolidados exportados a: {archivo_salida}")
print(f"  - Registros: {len(df_consolidado):,}")
print(f"  - Columnas: {len(df_consolidado.columns)}")
print(f"  - Tamaño: {archivo_salida.stat().st_size / 1024:.2f} KB")

# Exportar también en formato Excel para revisión manual
archivo_excel = DATA_PROCESSED_DIR / 'datos_consolidados_limpios.xlsx'
df_consolidado.to_excel(archivo_excel, index=False, engine='openpyxl')

print(f"\n✓ Datos también exportados en Excel: {archivo_excel}")


✓ Datos consolidados exportados a: C:\Users\Andres Diaz\Desktop\pipeline-proyecto-grado\data\processed\datos_consolidados_limpios.csv
  - Registros: 27
  - Columnas: 30
  - Tamaño: 6.55 KB

✓ Datos también exportados en Excel: C:\Users\Andres Diaz\Desktop\pipeline-proyecto-grado\data\processed\datos_consolidados_limpios.xlsx


---
## 7. REPORTE DE CALIDAD DE DATOS

In [7]:
# Generar reporte de calidad
reporte_calidad = f"""
{'='*80}
REPORTE DE CALIDAD DE DATOS - PROCESO ETL
{'='*80}

Fecha de generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

1. FUENTES DE DATOS PROCESADAS:
   - {', '.join(nombres_unis)}

2. ESTADÍSTICAS GENERALES:
   - Total de registros procesados: {len(df_consolidado):,}
   - Total de columnas: {len(df_consolidado.columns)}
   - Columnas con datos derivados: 5 (PERIODO, NOMBRE_COMPLETO, CATEGORIA_DURACION, FINANCIACION_TOTAL, TIENE_CONVENIO)

3. CALIDAD DE DATOS:
   - Porcentaje promedio de completitud: {completitud['Porcentaje_Completitud'].mean():.2f}%
   - Columnas con 100% completitud: {(completitud['Porcentaje_Completitud'] == 100).sum()}
   - Columnas con <50% completitud: {(completitud['Porcentaje_Completitud'] < 50).sum()}

4. TRANSFORMACIONES APLICADAS:
   ✓ Eliminación de filas vacías
   ✓ Limpieza de espacios en blanco
   ✓ Estandarización de nombres de países
   ✓ Conversión de tipos de datos
   ✓ Eliminación de duplicados
   ✓ Creación de variables derivadas

5. VALIDACIONES PASADAS:
   {chr(10).join(['   ' + v for v in validaciones])}

6. ARCHIVOS GENERADOS:
   - CSV: {archivo_salida}
   - Excel: {archivo_excel}

{'='*80}
DATOS LISTOS PARA ANÁLISIS DESCRIPTIVO Y PREDICTIVO
{'='*80}
"""

# Guardar reporte
archivo_reporte = REPORTS_DIR / 'reporte_calidad_datos.txt'
with open(archivo_reporte, 'w', encoding='utf-8') as f:
    f.write(reporte_calidad)

print(reporte_calidad)
print(f"\n✓ Reporte guardado en: {archivo_reporte}")


REPORTE DE CALIDAD DE DATOS - PROCESO ETL

Fecha de generación: 2025-10-31 01:31:17

1. FUENTES DE DATOS PROCESADAS:
   - IUSH

2. ESTADÍSTICAS GENERALES:
   - Total de registros procesados: 27
   - Total de columnas: 30
   - Columnas con datos derivados: 5 (PERIODO, NOMBRE_COMPLETO, CATEGORIA_DURACION, FINANCIACION_TOTAL, TIENE_CONVENIO)

3. CALIDAD DE DATOS:
   - Porcentaje promedio de completitud: 86.67%
   - Columnas con 100% completitud: 26
   - Columnas con <50% completitud: 4

4. TRANSFORMACIONES APLICADAS:
   ✓ Eliminación de filas vacías
   ✓ Limpieza de espacios en blanco
   ✓ Estandarización de nombres de países
   ✓ Conversión de tipos de datos
   ✓ Eliminación de duplicados
   ✓ Creación de variables derivadas

5. VALIDACIONES PASADAS:
      ✓ Años válidos (2020-2025): 27/27
   ✓ Semestres válidos (1-2): 27/27
   ✓ Días de movilidad positivos: 27/27

6. ARCHIVOS GENERADOS:
   - CSV: C:\Users\Andres Diaz\Desktop\pipeline-proyecto-grado\data\processed\datos_consolidados_lim

---
## 8. VISTA PREVIA DE DATOS LIMPIOS

In [9]:
print("\nVISTA PREVIA DE DATOS LIMPIOS:")
print("="*80)
display(df_consolidado.head(10))

print("\nINFORMACIÓN DEL DATAFRAME:")
print("="*80)
print(df_consolidado.info())

print("\nESTADÍSTICAS DESCRIPTIVAS:")
print("="*80)
display(df_consolidado.describe())


VISTA PREVIA DE DATOS LIMPIOS:


,AÑO,SEMESTRE,ID_TIPO_DOCUMENTO,NUM_DOCUMENTO,PRIMER_NOMBRE,SEGUNDO_NOMBRE,PRIMER_APELLIDO,SEGUNDO_APELLIDO,PAIS_EXTRANJERO,ID_PAIS_EXTRANJERO,...,ID_FUENTE_INTERNACIONAL,PAIS_FINANCIADOR,ID_PAIS_FINANCIADOR,VALOR_FINANCIACION_INTERNAC,PERIODO,NOMBRE_COMPLETO,CATEGORIA_DURACION,FINANCIACION_TOTAL,TIENE_CONVENIO,UNIVERSIDAD
0,2025.0,1.0,PS,G32986297,Jesús,Eduardo,Martínez,López,México,484.0,...,7.0,MÉXICO,484.0,5315000.0,2025.0-1.0,Jesús Eduardo Martínez López,Corta (≤7 días),5315000.0,False,IUSH
1,2025.0,1.0,PS,U32610501,Arzu,nan,Kirayoglu,nan,Turquía,792.0,...,7.0,ESPAÑA,724.0,5315000.0,2025.0-1.0,Arzu nan Kirayoglu nan,Corta (≤7 días),5315000.0,False,IUSH
2,2025.0,1.0,PS,N04903220,Daphne,Melissa,Cabrera,Tapia,México,484.0,...,7.0,MÉXICO,484.0,5315000.0,2025.0-1.0,Daphne Melissa Cabrera Tapia,Corta (≤7 días),5315000.0,False,IUSH
3,2025.0,1.0,PS,64640289,Catalin,nan,Serbu,nan,Rumania,642.0,...,7.0,ESPAÑA,724.0,5315000.0,2025.0-1.0,Catalin nan Serbu nan,Corta (≤7 días),5315000.0,False,IUSH
4,2025.0,1.0,PS,PAP274959,Luis,José,Callarisa,Fiol,España,724.0,...,7.0,ESPAÑA,724.0,5315000.0,2025.0-1.0,Luis José Callarisa Fiol,Corta (≤7 días),5315000.0,False,IUSH
5,2025.0,1.0,PS,AT8181419,Ioanna,nan,Papadaki,nan,Grecia,300.0,...,7.0,GRECIA,300.0,5315000.0,2025.0-1.0,Ioanna nan Papadaki nan,Corta (≤7 días),5315000.0,False,IUSH
6,2025.0,1.0,PS,A81664118,Stephanie,Aline,Morales,nan,Estados Unidos,840.0,...,7.0,MÉXICO,484.0,5315000.0,2025.0-1.0,Stephanie Aline Morales nan,Corta (≤7 días),5315000.0,False,IUSH
7,2025.0,1.0,PS,PAO684528,Cristina,nan,Díaz,Dobarro,España,724.0,...,7.0,ESPAÑA,724.0,5315000.0,2025.0-1.0,Cristina nan Díaz Dobarro,Corta (≤7 días),5315000.0,False,IUSH
8,2025.0,1.0,PS,N16400418,Alberto,Carlos,Becerra,Lopez,México,484.0,...,7.0,MÉXICO,484.0,5315000.0,2025.0-1.0,Alberto Carlos Becerra Lopez,Corta (≤7 días),5315000.0,False,IUSH
9,2025.0,1.0,PS,AY1243075,Vasiliki,nan,Nakou,nan,Grecia,300.0,...,7.0,GRECIA,300.0,5315000.0,2025.0-1.0,Vasiliki nan Nakou nan,Corta (≤7 días),5315000.0,False,IUSH



INFORMACIÓN DEL DATAFRAME:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   AÑO                          27 non-null     float64
 1   SEMESTRE                     27 non-null     float64
 2   ID_TIPO_DOCUMENTO            27 non-null     object 
 3   NUM_DOCUMENTO                27 non-null     object 
 4   PRIMER_NOMBRE                27 non-null     object 
 5   SEGUNDO_NOMBRE               27 non-null     object 
 6   PRIMER_APELLIDO              27 non-null     object 
 7   SEGUNDO_APELLIDO             27 non-null     object 
 8   PAIS_EXTRANJERO              27 non-null     object 
 9   ID_PAIS_EXTRANJERO           27 non-null     float64
 10  INSTITUCION_EXTRANJERA       27 non-null     object 
 11  TIPO_MOV_EST_EXTRANJ         27 non-null     object 
 12  ID_TIPO_MOV_EST_EXTRANJ      27 non-null     float64

,AÑO,SEMESTRE,ID_PAIS_EXTRANJERO,ID_TIPO_MOV_EST_EXTRANJ,NUM_DIAS_MOVILIDAD,CODIGO_CONVENIO,FUENTE_NACIONAL_INVESTIG,ID_FUENTE_NACIONAL_INVESTIG,VALOR_FINANCIACION_NACIONAL,ID_FUENTE_INTERNACIONAL,ID_PAIS_FINANCIADOR,VALOR_FINANCIACION_INTERNAC,FINANCIACION_TOTAL
count,27.0,27.0,27.000000,27.0,27.0,0.0,0.0,0.0,0.0,27.0,27.000000,27.0,27.0
mean,2025.0,1.0,432.444444,3.0,7.0,NaN,NaN,NaN,NaN,7.0,390.666667,5315000.0,5315000.0
std,0.0,0.0,225.499332,0.0,0.0,NaN,NaN,NaN,NaN,0.0,201.043432,0.0,0.0
min,2025.0,1.0,76.000000,3.0,7.0,NaN,NaN,NaN,NaN,7.0,76.000000,5315000.0,5315000.0
25%,2025.0,1.0,300.000000,3.0,7.0,NaN,NaN,NaN,NaN,7.0,300.000000,5315000.0,5315000.0
50%,2025.0,1.0,380.000000,3.0,7.0,NaN,NaN,NaN,NaN,7.0,380.000000,5315000.0,5315000.0
75%,2025.0,1.0,484.000000,3.0,7.0,NaN,NaN,NaN,NaN,7.0,484.000000,5315000.0,5315000.0
max,2025.0,1.0,862.000000,3.0,7.0,NaN,NaN,NaN,NaN,7.0,724.000000,5315000.0,5315000.0


---
**Fin del Notebook 2**

**Datos procesados y listos para:**
- Análisis descriptivo (Notebook 3)
- Modelos predictivos (Notebook 4)
- Integración con Power BI (Notebook 5)

Continuar con: `03_analisis_descriptivo.ipynb`